In [15]:
import pandas as pd
import sklearn
import xgboost
import matplotlib.pyplot as plt
import numpy as np


In [16]:
dataset = pd.read_excel('Dataset_Final.xlsx', sheet_name='Dataset_114')
wilayah_miskin = dataset # Buat pisahin dataset wilayah miskin ama indo
wilayah_miskin.shape

(1026, 9)

In [17]:
var = ["PPM", "TPAK", "TPT", "PDRB", "KP"]
X_prediktor = wilayah_miskin[var]
Y_target = wilayah_miskin["IPM"]
# Y_target.head(10)

In [18]:
from sklearn.model_selection import GroupShuffleSplit
data_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
index_latih, index_uji = next(data_split.split(X_prediktor, Y_target, groups=wilayah_miskin["Kabupaten/Kota"]))
data_latih = wilayah_miskin.iloc[index_latih]
data_uji = wilayah_miskin.iloc[index_uji]
data_latih.shape, data_uji.shape

((819, 9), (207, 9))

In [19]:
# daerah_uji = data_uji[["Kabupaten/Kota"]].drop_duplicates()
# daerah_uji.to_csv("daerah_uji.csv", index=False)
# print("Jumlah daerah uji:", len(daerah_uji))

In [20]:
# Buat baseline pake rata rata (nanti dipake juga buat pred awal)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_latih = data_latih[var]
y_latih = data_latih["IPM"]
x_uji = data_uji[var]
y_uji = data_uji["IPM"]

rata_rata = y_latih.mean()
prediksi_rata_rata = np.full(len(y_uji), rata_rata)

mae_bl = mean_absolute_error(y_uji, prediksi_rata_rata)
rmse_bl = np.sqrt(mean_squared_error(y_uji, prediksi_rata_rata))
r2_bl = r2_score(y_uji, prediksi_rata_rata)

print("Baseline Metrics:")
print("Mean Absolute Error (MAE):", mae_bl)
print("Root Mean Squared Error (RMSE):", rmse_bl)
print("R-squared (R2):", r2_bl)

Baseline Metrics:
Mean Absolute Error (MAE): 6.499822512431207
Root Mean Squared Error (RMSE): 8.390235601771394
R-squared (R2): -0.018579205435741653


In [ ]:
from xgboost import XGBRegressor

# Hyperparameter
model = XGBRegressor(
    n_estimators=1000,            # K   : jumlah pohon
    max_depth=2,                  # kedalaman pohon
    learning_rate=0.01,           # learning rate
    reg_lambda=1,                 # lambda: regularisasi pada bobot leaf
    reg_alpha=0,                  # regularisasi L1, tidak dipakai
    gamma=0,                      # gamma : minimum gain buat split
    objective="reg:squarederror", # loss function = 1/2 (y - yhat)^2
    base_score=float(y_latih.mean()),  # yhat(0) = pediksi awal
    random_state=0,
)

# latih model
model.fit(X_latih, y_latih)

# Buat prediksi untuk data latih dan data uji
pred_latih = model.predict(X_latih)
pred_uji = model.predict(x_uji)

# evaluasi model
def hitung_metrik(y_asli, y_pred):
    return {
        "MAE": mean_absolute_error(y_asli, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_asli, y_pred)),
        "R2": r2_score(y_asli, y_pred),
    }

# buat tabel hasil evaluasi
hasil_metrik = pd.DataFrame({
    "Baseline (uji)": hitung_metrik(y_uji, prediksi_rata_rata),
    "Model (latih)": hitung_metrik(y_latih, pred_latih),
    "Model (uji)": hitung_metrik(y_uji, pred_uji),
}).T.round(3)
hasil_metrik

,MAE,RMSE,R2
Baseline (uji),6.500,8.390,-0.019
Model (latih),2.056,2.586,0.868
Model (uji),2.766,3.654,0.807


: 